# VAE Evaluation — Extended Features vs. Original (CC1 + CC2 only)

**New notebook.** Evaluates the extended-feature model (`train_vae_extended.ipynb`)
with the exact same leak-free protocol as `vae_eval.ipynb`, and prints a direct
side-by-side comparison against the original 7-feature model's numbers — this
is the notebook that actually answers "did adding features help?".

In [1]:
import numpy as np
import torch
import torch.nn as nn
import pickle, os
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix,
)

BASE          = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR      = os.path.join(BASE, 'experiments', 'models_extended', 'data', 'processed_extended', 'windows_cc1_extended')
MODEL_DIR_EXT = os.path.join(BASE, 'experiments', 'models_extended', 'model')
MODEL_DIR_ORIG = os.path.join(BASE, 'models')

ALL_SETS = ['cc1_test', 'drift_cc2']

class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

meta = pickle.load(open(os.path.join(MODEL_DIR_EXT, 'vae_cc1_extended_meta.pkl'), 'rb'))
print('Extended model metadata:', {k: v for k, v in meta.items() if k not in ('beta_search', 'ablation')})

model = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
model.load_state_dict(torch.load(os.path.join(MODEL_DIR_EXT, 'vae_cc1_extended.pt'), map_location='cpu'))
model.eval()
CLIP = meta['clip']

Extended model metadata: {'input_dim': 26, 'hidden1': 64, 'hidden2': 32, 'latent_dim': 8, 'beta_max': 0.1, 'clip': 20.0, 'mu_train': 0.7372125387191772, 'sigma_train': 0.580580472946167, 'feature_cols': ['container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate', 'container_cpu_cfs_throttled_seconds_rate', 'container_cpu_cfs_throttled_periods_rate', 'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache', 'container_threads', 'memory_limit_proximity']}


## Step 1 — Recompute scores, leak-free threshold from `cc1_val`

In [2]:
X, y, mse = {}, {}, {}
for name in ['cc1_train', 'cc1_val'] + ALL_SETS:
    Xr = np.load(os.path.join(DATA_DIR, f'X_{name}.npy'))
    X[name] = np.clip(Xr, -CLIP, CLIP).astype(np.float32)
    y[name] = np.load(os.path.join(DATA_DIR, f'y_{name}.npy'))
    mse[name] = model.anomaly_score(torch.from_numpy(X[name])).numpy()
    print(f'  {name:10s}: {len(y[name]):>7,} windows  |  anomalies: {int(y[name].sum()):,}')

assert abs(mse['cc1_train'].mean() - meta['mu_train']) < 1e-3
val_p99 = float(np.percentile(mse['cc1_val'], 99))
print(f'\nval_p99 threshold (extended model): {val_p99:.5f}')

  cc1_train : 154,198 windows  |  anomalies: 0
  cc1_val   :  21,573 windows  |  anomalies: 0
  cc1_test  :  44,185 windows  |  anomalies: 256
  drift_cc2 :  76,977 windows  |  anomalies: 720

val_p99 threshold (extended model): 2.54080


## Step 2 — Extended model results

In [3]:
def evaluate(scores, y_true, threshold):
    pred = (scores > threshold).astype(int)
    return {
        'auc_roc': roc_auc_score(y_true, scores), 'auc_pr': average_precision_score(y_true, scores),
        'precision': precision_score(y_true, pred, zero_division=0), 'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
    }

extended_results = {name: evaluate(mse[name], y[name], val_p99) for name in ALL_SETS}
print(f'{"set":12s} {"PR-AUC":>8s} {"ROC-AUC":>9s} {"F1":>7s} {"Precision":>10s} {"Recall":>8s}')
for name in ALL_SETS:
    r = extended_results[name]
    print(f'{name:12s} {r["auc_pr"]:8.4f} {r["auc_roc"]:9.4f} {r["f1"]:7.3f} {r["precision"]:10.3f} {r["recall"]:8.3f}')

set            PR-AUC   ROC-AUC      F1  Precision   Recall
cc1_test       0.3924    0.8669   0.421      0.500    0.363
drift_cc2      0.0345    0.7073   0.054      0.029    0.511


## Step 3 — Direct comparison: extended (11 features) vs. original (7 features)

In [4]:
orig_eval = pickle.load(open(os.path.join(MODEL_DIR_ORIG, 'vae_cc1_eval.pkl'), 'rb'))

print(f'{"set":12s} {"model":10s} {"PR-AUC":>8s} {"ROC-AUC":>9s} {"F1":>7s} {"Precision":>10s} {"Recall":>8s}')
for name in ALL_SETS:
    ext = extended_results[name]
    orig_pr = orig_eval['precision_recall'][name]['val_p99']
    orig_auc = orig_eval['auc'][name]
    print(f'{name:12s} {"extended":10s} {ext["auc_pr"]:8.4f} {ext["auc_roc"]:9.4f} {ext["f1"]:7.3f} {ext["precision"]:10.3f} {ext["recall"]:8.3f}')
    print(f'{name:12s} {"original":10s} {orig_auc["auc_pr"]:8.4f} {orig_auc["auc_roc"]:9.4f} {orig_pr["f1"]:7.3f} {orig_pr["precision"]:10.3f} {orig_pr["recall"]:8.3f}')
    print(f'  -> PR-AUC change: {ext["auc_pr"] - orig_auc["auc_pr"]:+.4f}   F1 change: {ext["f1"] - orig_pr["f1"]:+.3f}\n')

set          model        PR-AUC   ROC-AUC      F1  Precision   Recall
cc1_test     extended     0.3924    0.8669   0.421      0.500    0.363
cc1_test     original     0.6014    0.8763   0.618      0.630    0.605
  -> PR-AUC change: -0.2090   F1 change: -0.197

drift_cc2    extended     0.0345    0.7073   0.054      0.029    0.511
drift_cc2    original     0.4089    0.8812   0.276      0.168    0.772
  -> PR-AUC change: -0.3744   F1 change: -0.222



## Step 4 — Per-fault-type recall: does `pod-failure` specifically improve? (the whole point of `container_threads`)

In [5]:
ft = {name: np.load(os.path.join(DATA_DIR, f'ft_{name}.npy'), allow_pickle=True) for name in ALL_SETS}
orig_fault = orig_eval['per_fault_recall']

print(f'{"set":12s} {"fault_type":14s} {"n":>5s} {"original recall":>16s} {"extended recall":>16s}')
for name in ALL_SETS:
    pred = (mse[name] > val_p99).astype(int)
    types_present = sorted({v for v in ft[name] if isinstance(v, str)})
    for ftype in types_present:
        mask = ft[name] == ftype
        n = int(mask.sum())
        rec_ext = pred[mask].mean() if n > 0 else float('nan')
        rec_orig = orig_fault[name][ftype]['recall']
        print(f'{name:12s} {ftype:14s} {n:5d} {rec_orig:16.3f} {rec_ext:16.3f}')
    print()

set          fault_type         n  original recall  extended recall
cc1_test     cpu               88            0.580            0.307
cc1_test     memory            76            0.763            0.645
cc1_test     pod-failure       92            0.500            0.185

drift_cc2    cpu              207            0.908            0.652
drift_cc2    memory           414            0.713            0.522
drift_cc2    pod-failure       99            0.737            0.172



## Step 5 — Save

In [6]:
out_path = os.path.join(MODEL_DIR_EXT, 'vae_cc1_extended_eval.pkl')
with open(out_path, 'wb') as f:
    pickle.dump({'results': extended_results, 'val_p99': val_p99}, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\models_extended\vae_cc1_extended_eval.pkl


## How to read this

Step 3 is the direct answer to "did feature engineering help?" — compare the
PR-AUC and F1 change rows. Step 4 specifically tests the `container_threads`
hypothesis: if `pod-failure` recall improved meaningfully (it was the weakest
fault type at 0.50/0.737 in-distribution/drift with the original 7 features),
that's evidence the added feature earned its place. If CPU recall didn't move
much, that's consistent with the throttling features being uninformative here
(verified earlier — always zero in this dataset).